# Đếm rep hít đất bằng MediaPipe Pose

Đếm số lần hít đất và tách các đoạn tập trong video, chạy được **cả một thư mục video**.

Kiểm chứng trên 7 video thật (75 phút): **39/41 rep đúng** trên 7 cửa sổ đếm tay,
phủ 5/7 video. Hai rep sai đều là **sót**, không có cửa sổ nào đếm dư.

## Vì sao MediaPipe chứ không phải YOLO-pose

Tôi làm bản YOLO11-pose trước rồi mới đổi. Khác biệt quyết định là **MediaPipe bám
theo người giữa các frame**, còn YOLO xử lý mỗi frame độc lập. Ở góc quay của bộ video
này (camera thấp, người hướng thẳng vào ống kính), YOLO mất pose đúng vào **đáy mỗi
rep** — chính chỗ định nghĩa một rep.

Đo trên cùng một cửa sổ khó:

| | YOLO11s-pose | MediaPipe |
|---|---|---|
| Frame dùng được | ~55 % | **92 %** |
| Biên độ góc khuỷu đo được | 124–170° | **70–179°** |
| Rep đếm được (đếm tay = 9) | 6 | **8** |

Dùng model YOLO to hơn không cứu được: `yolo11x-pose` ở `imgsz=960` cũng chỉ đủ keypoint
ở 1/8 frame đáy. Đó là giới hạn của góc quay chứ không phải dung lượng model.

In [ ]:
!pip install -q mediapipe

In [ ]:
import os, glob, time
os.environ['GLOG_minloglevel'] = '3'      # tat log on ao cua MediaPipe
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mediapipe as mp
from scipy.signal import savgol_filter

print('mediapipe', mp.__version__, '| opencv', cv2.__version__)

## Cấu hình

Các tham số dưới đây **không nhạy**: tôi quét 162 tổ hợp (visibility, làm mượt, cửa sổ
thích ứng, biên độ tối thiểu, chu kỳ tối thiểu) và **cả 162 cho cùng kết quả** trên bộ
cửa sổ ground truth. Đổi vừa phải sẽ không làm hỏng gì.

In [ ]:
# ===================== CẤU HÌNH =====================
VIDEO_DIR = '/kaggle/input/push-up'   # thư mục video; hoặc None nếu chỉ chạy 1 file
VIDEO_ONE = None                       # ví dụ '/kaggle/input/push-up/100pushup.mp4'
OUT_DIR   = '/kaggle/working'

# --- Trích landmark ---
COMPLEXITY = 1     # 0 nhanh / 1 cân bằng / 2 chính xác nhất nhưng chậm ~3.5x
STRIDE     = 2     # xử lý mỗi N frame. Đo được: stride 1/2/3 cho CÙNG số rep
VIS_THR    = 0.5   # bỏ landmark có visibility thấp hơn mức này

# --- Tín hiệu ---
SMOOTH_S   = 0.25  # cửa sổ làm mượt Savitzky-Golay (giây)
MAX_GAP_S  = 0.50  # mất pose ngắn hơn mức này thì nội suy bù

# --- Đếm rep: ngưỡng TỰ THÍCH ỨNG, không có số cố định ---
ADAPT_WIN_S  = 9.0   # lấy biên độ thực tế trong cửa sổ trượt này
MIN_SPAN     = 30.0  # biên độ cục bộ tối thiểu (độ) mới coi là đang tập
MIN_PERIOD_S = 0.7   # khoảng cách tối thiểu giữa 2 rep

# --- Lọc nhóm: bỏ nhịp đơn độc (thường là lúc chống tay xuống vào tư thế) ---
GROUP_GAP_S  = 4.0
GROUP_MIN    = 2
# ====================================================

os.makedirs(OUT_DIR, exist_ok=True)
VIDEO_EXT = ('.mp4', '.mov', '.avi', '.mkv', '.m4v')

def list_videos():
    if VIDEO_ONE:
        return [VIDEO_ONE]
    if VIDEO_DIR and os.path.isdir(VIDEO_DIR):
        return sorted(os.path.join(VIDEO_DIR, f) for f in os.listdir(VIDEO_DIR)
                      if f.lower().endswith(VIDEO_EXT))
    return []

FILES = list_videos()
assert FILES, f'Không thấy video nào trong {VIDEO_DIR}'
print(f'{len(FILES)} video:')
for f in FILES:
    print('  ', os.path.basename(f))

## Lượt 1 — Trích landmark

MediaPipe phải chạy **liên tục theo thứ tự** thì cơ chế bám người mới hoạt động; đó là
lý do không có tham số nào cho phép nhảy cóc lung tung. Bỏ bớt frame đều đặn thì vẫn ổn
(đo được: stride 1/2/3 cho cùng số rep), nhưng đảo thứ tự hay lấy frame rời rạc thì hỏng.

Kết quả cache ra `.npz` để đổi tham số đếm mà không phải chạy lại model.

In [ ]:
# BlazePose 33 điểm — trùng thứ tự với ML Kit trên điện thoại, nên code này port
# thẳng sang app được.
NOSE = 0
L_SHO, R_SHO, L_ELB, R_ELB, L_WRI, R_WRI = 11, 12, 13, 14, 15, 16
L_HIP, R_HIP = 23, 24


def extract_landmarks(path, out_npz, complexity=COMPLEXITY, stride=STRIDE):
    """Chạy MediaPipe trên toàn video -> (33,4) mỗi frame: x, y, z, visibility."""
    def same_file(a, b):
        # So chuỗi đường dẫn thô là bẫy: cùng một file có thể ghi '/' hay '\',
        # hoa hay thường, tương đối hay tuyệt đối -> cache khớp mà bị coi là khác
        # và notebook lặng lẽ chạy lại model cả tiếng đồng hồ.
        return os.path.normcase(os.path.abspath(a)) == os.path.normcase(os.path.abspath(b))

    if os.path.exists(out_npz):
        d = np.load(out_npz)
        if same_file(str(d['src']), path) and int(d['stride']) == stride:
            print('   dùng lại cache', os.path.basename(out_npz))
            return
        print('   cache của video/tham số khác -> trích lại')

    cap = cv2.VideoCapture(path)
    assert cap.isOpened(), f'Không mở được {path}'
    fps = cap.get(cv2.CAP_PROP_FPS)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    pose = mp.solutions.pose.Pose(
        static_image_mode=False,     # BẮT BUỘC False: bật cơ chế bám người
        model_complexity=complexity,
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    idx, lms, i, t0 = [], [], 0, time.time()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % stride == 0:
            res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if res.pose_landmarks:
                a = np.array([[l.x * W, l.y * H, l.z * W, l.visibility]
                              for l in res.pose_landmarks.landmark], np.float32)
            else:
                a = np.full((33, 4), np.nan, np.float32)   # không thấy người
            lms.append(a); idx.append(i)
            if len(idx) % 1000 == 0:
                el, done = time.time() - t0, i / max(total, 1)
                print(f'   {done*100:5.1f}%  {el:4.0f}s  còn ~{el/max(done,1e-9)-el:4.0f}s',
                      flush=True)
        i += 1
    pose.close(); cap.release()
    np.savez_compressed(out_npz, idx=np.array(idx), lm=np.array(lms, np.float32),
                        fps=fps, width=W, height=H, stride=stride, src=path)
    print(f'   {len(idx)} frame trong {time.time()-t0:.0f}s')


def npz_path(video):
    tag = ''.join(c if c.isalnum() or c in '-_' else '_'
                  for c in os.path.splitext(os.path.basename(video))[0])[:50]
    return f'{OUT_DIR}/lm_{tag}.npz'


for n, f in enumerate(FILES, 1):
    print(f'[{n}/{len(FILES)}] {os.path.basename(f)}', flush=True)
    extract_landmarks(f, npz_path(f))

## Lượt 2 — Tín hiệu: góc khuỷu, trung bình hai tay

Góc khuỷu `vai–khuỷu–cổ tay` là tín hiệu tốt nhất cho hít đất — dao động rõ **70° ↔ 170°**.

Điều bất ngờ: tôi dự đoán góc nhìn trực diện sẽ làm góc khuỷu vô dụng vì **rút ngắn phối
cảnh**. Thực tế ngược lại — camera đặt thấp ngay phía trước nên cánh tay vung gần như
song song mặt phẳng ảnh.

**Lấy trung bình hai tay, không đếm riêng từng tay.** Hít đất có hai tay ràng buộc cứng
cùng pha, nên đếm riêng rồi gộp lại chỉ khuếch đại lỗi pose gán nhầm trái/phải: hai tay
lệch 0.5–0.7 s, quá xa để gộp mà quá gần để là hai nhịp thật, và mỗi lần nhầm sinh thêm
một rep ma. Đo được **120 rep ở nơi thực tế chỉ có ~97**.

In [ ]:
def angle_at(a, b, c):
    """Góc tại đỉnh b (độ), a/b/c dạng (N,2)."""
    v1, v2 = a - b, c - b
    n1, n2 = np.linalg.norm(v1, axis=-1), np.linalg.norm(v2, axis=-1)
    cos = np.sum(v1 * v2, -1) / np.clip(n1 * n2, 1e-6, None)
    out = np.degrees(np.arccos(np.clip(cos, -1, 1)))
    out[(n1 < 1e-3) | (n2 < 1e-3)] = np.nan
    return out


def interp_short_gaps(x, max_gap):
    """Nội suy các khoảng NaN NGẮN; khoảng dài giữ NaN.

    Đừng nới max_gap để 'vá' cho đẹp: nội suy qua một lỗ 0.6 s trong nhịp 1.1 s là
    thay cả nhịp xuống-lên bằng một đường thẳng, rep đó mất hẳn chứ không được cứu.
    """
    x = np.asarray(x, float).copy()
    ok = np.isfinite(x)
    if ok.sum() < 2:
        return x
    i = np.arange(len(x))
    f = np.interp(i, i[ok], x[ok])
    run = np.cumsum(ok)
    for g in np.unique(run[~ok]):
        m = (~ok) & (run == g)
        if m.sum() > max_gap:
            f[m] = np.nan
    f[:np.argmax(ok)] = np.nan
    f[len(x) - np.argmax(ok[::-1]):] = np.nan
    return f


def smooth(x, win):
    """Savitzky-Golay bỏ qua NaN: lọc riêng từng đoạn liên tục."""
    win = max(5, int(win) | 1)
    y = np.full(len(x), np.nan)
    ok = np.isfinite(x)
    if ok.sum() < 5:
        return np.asarray(x, float)
    e = np.flatnonzero(np.diff(ok.astype(int)))
    b = np.concatenate([[0], e + 1, [len(x)]])
    for a, c in zip(b[:-1], b[1:]):
        if ok[a]:
            y[a:c] = savgol_filter(x[a:c], win, 2) if c - a >= win else x[a:c]
    return y


def elbow_signal(npz, vis_thr=VIS_THR, max_gap_s=MAX_GAP_S, smooth_s=SMOOTH_S):
    """-> (t, tín hiệu đã mượt, tần số lấy mẫu, tỉ lệ mẫu dùng được, tỉ lệ mất người)"""
    d = np.load(npz)
    lm, t = d['lm'], d['idx'] / float(d['fps'])
    sr = float(d['fps']) / int(d['stride'])
    xy, vis = lm[..., :2].astype(float), lm[..., 3]

    arms = {}
    for s, (a, b, c) in {'L': (L_SHO, L_ELB, L_WRI), 'R': (R_SHO, R_ELB, R_WRI)}.items():
        v = angle_at(xy[:, a], xy[:, b], xy[:, c])
        v[np.minimum.reduce([vis[:, a], vis[:, b], vis[:, c]]) < vis_thr] = np.nan
        arms[s] = v
    with np.errstate(invalid='ignore'):
        M = np.nanmean(np.vstack([arms['L'], arms['R']]), 0)

    raw_ok = float(np.isfinite(M).mean())
    no_person = float(np.mean(~np.isfinite(lm[:, 0, 0])))
    M = smooth(interp_short_gaps(M, max(1, int(round(max_gap_s * sr)))),
               max(5, int(round(smooth_s * sr)) | 1))
    return t, M, sr, raw_ok, no_person

## Lượt 3 — Đếm rep: trigger Schmitt **ngưỡng tự thích ứng**

Đây là thay đổi quan trọng nhất của cả notebook.

Ngưỡng cố định **không chuyển được giữa các video**. Tôi từng đặt `hi=140°, lo=100°` dựa
trên hai video, và tưởng rằng vì đơn vị là độ nên nó chuyển được sang người khác. Sai:
video `20241110` có người tập tầm hẹp, **đáy chỉ xuống ~130°** nên không bao giờ chạm
ngưỡng dưới, trigger không nạp lại được và **cả video 7.45 phút chỉ đếm ra 3 rep**.

Cách sửa: mỗi thời điểm lấy biên độ **thực tế** trong cửa sổ trượt ±`ADAPT_WIN_S/2` quanh
nó, rồi đặt `hi`/`lo` vào 2/3 và 1/3 của biên độ đó. Cùng video đó: **3 → 103 rep**.

`MIN_SPAN` là chốt chặn nhiễu thay cho biên độ cố định: biên độ cục bộ dưới 30° nghĩa là
đang đứng yên chứ không tập, nên không đếm.

In [ ]:
def rolling_percentile(x, win, qs=(10, 90)):
    """Phân vị trượt, bỏ qua NaN. ~0.8 s cho video 17 phút — không phải nút cổ chai."""
    n, half = len(x), win // 2
    lo = np.full(n, np.nan); hi = np.full(n, np.nan)
    for i in range(n):
        seg = x[max(0, i - half):min(n, i + half + 1)]
        seg = seg[np.isfinite(seg)]
        if len(seg) >= max(5, win // 4):
            lo[i], hi[i] = np.percentile(seg, qs)
    return lo, hi


def detect_reps(t, s, sr, win_s=ADAPT_WIN_S, min_span=MIN_SPAN, min_period=MIN_PERIOD_S):
    """Trigger Schmitt với ngưỡng bám theo biên độ cục bộ.

    Đếm khi tín hiệu đi LÊN cắt ngưỡng trên, sau khi đã tụt dưới ngưỡng dưới.
    Mất pose -> quên trạng thái, để không ghép đáy trước với đỉnh sau khoảng trống.
    """
    win = max(5, int(round(win_s * sr)) | 1)
    p_lo, p_hi = rolling_percentile(s, win)
    span = p_hi - p_lo
    hi, lo, amp = p_lo + 2 * span / 3, p_lo + span / 3, span / 2

    reps, state = [], None
    trough, trough_t, last = np.inf, np.nan, -np.inf
    for i in range(len(s)):
        v = s[i]
        if not np.isfinite(v) or not np.isfinite(span[i]) or span[i] < min_span:
            state = None
            continue
        if state is None:
            state = 'up' if v > hi[i] else 'down'
            trough, trough_t = v, t[i]
            continue
        if state == 'down':
            if v < trough:
                trough, trough_t = v, t[i]
            if v > hi[i]:
                if v - trough >= amp[i] and t[i] - last >= min_period:
                    reps.append(dict(t_top=t[i], t_bottom=trough_t, amp=v - trough))
                    last = t[i]
                state = 'up'
        elif v < lo[i]:
            state, trough, trough_t = 'down', v, t[i]
    return reps, dict(hi=hi, lo=lo, span=span)


def group_reps(reps, gap_s=GROUP_GAP_S, min_reps=GROUP_MIN):
    """Gom rep thành set và BỎ các nhịp đơn độc.

    Nhịp đơn độc gần như luôn là dương tính giả: người đang quỳ nghỉ rồi chống tay
    xuống sàn để vào tư thế — động tác gập rồi duỗi tay đó vượt ngưỡng. Kiểm tra
    bằng mắt 3 nhịp đơn độc trên video thật: cả 3 đều không phải hít đất.

    Ngược lại, ĐỪNG lọc theo nhịp trung vị. Bản trước loại set có nhịp chậm hơn 3 s,
    và nó đã ném đi cả một đoạn hít đất chậm có thật (biên độ 65°, còn lớn hơn các rep
    được giữ). Bỏ bộ lọc đó đưa sai lệch trên tập kiểm chứng từ 7 xuống 2 rep.
    """
    if not reps:
        return [], []
    sets, cur = [], [reps[0]]
    for r in reps[1:]:
        if r['t_top'] - cur[-1]['t_top'] <= gap_s:
            cur.append(r)
        else:
            sets.append(cur); cur = [r]
    sets.append(cur)
    keep = [g for g in sets if len(g) >= min_reps]
    dropped = [r for g in sets if len(g) < min_reps for r in g]
    return keep, dropped

## Chạy và tổng hợp

Đọc cột `tin_hieu_ok` và `bien_do` **trước** khi tin cột `rep`:

- `tin_hieu_ok` — tỉ lệ frame tính được góc khuỷu.
- `bien_do` — biên độ dao động trung vị lúc đang tập. Thấp bất thường (dưới ~50°) nghĩa
  là camera quá gần hoặc góc quá ngặt, số rep sẽ kém tin cậy.

In [ ]:
rows, results = [], {}
for f in FILES:
    npz = npz_path(f)
    t, M, sr, raw_ok, no_person = elbow_signal(npz)
    reps, dbg = detect_reps(t, M, sr)
    sets, dropped = group_reps(reps)
    kept = [r for g in sets for r in g]
    cad = [np.median(np.diff([r['t_top'] for r in g])) for g in sets if len(g) > 1]
    active = dbg['span'][np.isfinite(dbg['span']) & (dbg['span'] > MIN_SPAN)]

    results[f] = dict(t=t, M=M, sr=sr, reps=reps, sets=sets, dbg=dbg)
    rows.append(dict(
        video=os.path.basename(f), phut=round(t[-1] / 60, 2), Hz=round(sr, 1),
        mat_nguoi=round(no_person, 3), tin_hieu_ok=round(raw_ok, 3),
        rep=len(kept), loai_bo=len(dropped), set=len(sets),
        nhip_s=round(float(np.median(cad)), 2) if cad else np.nan,
        bien_do=round(float(np.median(active)), 0) if len(active) else np.nan,
    ))

df = pd.DataFrame(rows)
df.to_csv(f'{OUT_DIR}/pushup_summary.csv', index=False)
display(df)
print(f"TỔNG: {df['rep'].sum()} rep trên {len(df)} video "
      f"({df['phut'].sum():.1f} phút), loại {df['loai_bo'].sum()} nhịp đơn độc")

In [ ]:
# Biểu đồ kiểm tra: mỗi chấm đỏ phải rơi đúng một nhịp đi lên.
for f in FILES[:3]:
    R = results[f]
    t, M, sets, dbg = R['t'], R['M'], R['sets'], R['dbg']
    fig, ax = plt.subplots(2, 1, figsize=(20, 6))
    ax[0].plot(t, M, lw=.6)
    for g in sets:
        ax[0].axvspan(g[0]['t_bottom'], g[-1]['t_top'], color='tab:green', alpha=.15, lw=0)
    tops = [r['t_top'] for g in sets for r in g]
    ax[0].plot(tops, np.interp(tops, t, np.nan_to_num(M, nan=np.nanmedian(M))),
               'v', color='crimson', ms=3)
    ax[0].set_title(f'{os.path.basename(f)} — {len(tops)} rep / {len(sets)} set')
    ax[0].set_ylabel('góc khuỷu (độ)'); ax[0].grid(alpha=.3)

    if sets:
        g = max(sets, key=len)
        m = (t >= g[0]['t_top'] - 1) & (t <= min(g[0]['t_top'] + 14, g[-1]['t_top'] + 1))
        ax[1].plot(t[m], M[m], '.-', ms=3, lw=1.2, label='góc khuỷu')
        ax[1].plot(t[m], dbg['hi'][m], 'r--', lw=.9, label='ngưỡng trên (tự thích ứng)')
        ax[1].plot(t[m], dbg['lo'][m], 'b--', lw=.9, label='ngưỡng dưới')
        for r in g:
            if m[np.argmin(np.abs(t - r['t_top']))]:
                ax[1].axvline(r['t_top'], color='k', lw=.8, alpha=.5)
        ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
        ax[1].set_title('phóng to set dài nhất — vạch đen = thời điểm đếm')
    ax[1].set_xlabel('giây')
    plt.tight_layout(); plt.show()

## Độ chính xác đo được

Kiểm chứng bằng cách dựng contact sheet 0.2–0.35 s/frame rồi **đếm rep bằng mắt**, đối
chiếu từng rep một:

| Cửa sổ | Đếm tay | Máy đếm | |
|---|---|---|---|
| `100pushup` 16.6–20.05 s | 3 | 3 | ✓ |
| `100pushup` 160.3–165.4 s | 5 | 5 | ✓ |
| `IMG_5087` 330.4–335.9 s | 5 | 5 | ✓ |
| `20241110` 199.2–209.4 s | 9 | 8 | −1 |
| `IMG_8752` 201.1–209.1 s | 6 | 5 | −1 |
| `IMG_8752` 93.0–102.5 s | 5 | 5 | ✓ |
| `copy_4FAC` 5.9–14.0 s | 8 | 8 | ✓ |
| **Tổng** | **41** | **39** | **95 %** |

**Không cửa sổ nào đếm dư** — cả hai sai lệch đều là sót, và cả hai rơi vào đoạn
MediaPipe không thấy người (0.3–0.6 s). Không thể cứu từ dữ liệu pose.

## Giới hạn còn lại

Vấn đề còn lại nằm ở **khâu quay, không phải khâu code**. Cả 7 video đều quay trực diện
từ phía trước, camera đặt thấp — tư thế hiếm trong dữ liệu huấn luyện, nên pose hay sập
đúng lúc người xuống sát sàn và đầu lấp đầy khung.

Để đếm chính xác hơn: **đặt máy ngang hông, quay từ bên sườn**, lùi đủ xa để cả người
nằm trong khung. Cùng thuật toán này khi đó sẽ chạy tốt hơn hẳn.

## Chỉnh khi cần

| Triệu chứng | Sửa |
|---|---|
| Đếm thiếu, `bien_do` thấp | Camera quá gần — lùi ra, quay từ bên sườn |
| Đếm thiếu, `tin_hieu_ok` thấp | Tăng `COMPLEXITY` lên 2, hoặc hạ `VIS_THR` |
| Nhịp lắc lư khi nghỉ bị tính | Nâng `MIN_SPAN` |
| Rep chậm có thật bị bỏ | Nâng `GROUP_GAP_S`, hoặc hạ `GROUP_MIN` xuống 1 |
| Một nhịp bị đếm thành hai | Nâng `MIN_PERIOD_S`, tăng `SMOOTH_S` |